# Milestone 2 · Session 1 of 3 — TimesFM 2.5: factor probes + fairness

Finishes the C-MAPSS vertical slice (IMPLEMENTATION_PLAN §5) together with its two sibling notebooks.

Session split (one backbone per runtime — `requirements/README.md`):

1. `timesfm_probes.ipynb` — **TimesFM 2.5**: RQ-A/C/E/H factor probes **with the shared baselines** + RQ-M fairness (TimesFM).
2. `chronos_probes_zeroshot.ipynb` — **Chronos-2**: the same probes models-only + RQ-M fairness (Chronos-2) + **RQ-Z zero-shot**.
3. `fairness_moment_ttm_moirai.ipynb` — RQ-M fairness for **MOMENT / TTM / Moirai-2** (one model per runtime cycle).

The probe roster (top-2 TSFMs TimesFM 2.5 + Chronos-2, foils gbm + minirocket, NN lstm) is the `probe_roster` resolution recorded in CHANGES.md §46.

**Before running:** `Runtime ▸ Change runtime type ▸ GPU`; use a **fresh runtime** (the backbones must never share an environment); run top-to-bottom. Every stage is restartable — re-running resumes and skips completed cells; levels marked *(cache hit)* reuse the §45 campaign caches on Drive.


In [ ]:
# 1) clone the repo (shallow) — same pattern as notebooks/campaign/* Stage A
%cd /content
!git clone --depth 1 --branch main https://github.com/blozanod/Predictive-Maintenance-LSTM.git 2>/dev/null || echo "(already cloned — reusing)"
%cd /content/Predictive-Maintenance-LSTM
import sys; sys.path.insert(0, '/content/Predictive-Maintenance-LSTM')   # import src.* from the fresh clone


In [ ]:
# 2) install ONLY TimesFM 2.5's isolated stack (see requirements/ for the pin rationale).
!pip install -r requirements/timesfm.txt


In [ ]:
# 3) mount Google Drive — the SAME folder the §45 campaign wrote the caches/results to
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 4) the CANONICAL config — identical to the §45 campaign notebooks for every cache-key
#    field (window / sensors / max_rul / pooling / context / condition_norm), so every
#    probe level that matches the campaign shape is a CACHE HIT on Drive.
from src.config import Config

DRIVE = '/content/drive/MyDrive/pdm_tsfm'   # SAME Drive folder as the §45 campaign
MODEL = 'google/timesfm-2.5-200m-pytorch'
TAG = 'timesfm'                     # per-session file suffix: two sessions never append to one Drive CSV
RESULTS = f'{DRIVE}/results'

config = Config(
    data_root='Data',                  # C-MAPSS is committed in the repo clone
    cache_dir=f'{DRIVE}/cache',        # the campaign embedding caches live here
    results_dir=RESULTS,
    model_name=MODEL,
    tsfm_context_length=256,           # recorded FD001 winner (CHANGES.md §12)
    pooling='mean',
    head_features='emb+locscale',      # head knob; not a cache key (CHANGES.md §9)
)

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device


## RQ-M — representation-fairness ablation (`run_representation_fairness`)

Each model runs TWICE at full data, MSE, 3 seeds (CHANGES.md §35): **native**
(`channel_aggregation='concat'`, its own pooling — the campaign shape, so this arm is a
**cache hit**) vs **common** (`channel_aggregation='mean'`, `pooling='mean'` — one new
Stage-A embedding pass per dataset). Anchors: **FD001** (single-condition) + **FD004**
(multi-condition contrast) — the cross-TSFM ranking is checked on both regimes.
Restartable; per-session CSV so parallel sessions never share a file.


In [ ]:
from src.sweep import run_representation_fairness

for ds in ['FD001', 'FD004']:
    p = run_representation_fairness(
        config.replace(dataset=ds, sensor_columns=None),
        models=[MODEL], device=device,
        out_csv=f'{RESULTS}/representation_fairness_{TAG}.csv')
    print('fairness →', p)


In [ ]:
# Probe roster (CHANGES.md §46 `probe_roster` resolution): foils gbm + minirocket,
# NN lstm, + the predict_mean floor for the hollow guard. Baselines run in THIS
# session only — the Chronos session runs models-only and scoring globs both files.
from src.probes import run_factor_probe

BASELINES = ['gbm', 'minirocket', 'lstm', 'predict_mean']


## RQ-H — noise tolerance (sim-only, FD001 anchor per RESEARCH_PLAN §5)

`noise_injection` (CHANGES.md §38) perturbs the SIMULATED readings after labels, before
windowing — baselines see the same degraded windows. Levels: the clean reference
*(cache hit)*, gaussian at 30/20/10 dB SNR, a 1-channel-std calibration drift, and 10 %
sensor dropout. Each non-clean level builds one new FD001 cache (embed pass on GPU).


In [ ]:
NOISE_LEVELS = {
    'clean':     {},                                    # cache hit — unperturbed reference
    'snr30':     {'kind': 'gaussian', 'snr_db': 30.0},
    'snr20':     {'kind': 'gaussian', 'snr_db': 20.0},
    'snr10':     {'kind': 'gaussian', 'snr_db': 10.0},
    'drift1':    {'kind': 'drift',    'magnitude': 1.0},
    'dropout10': {'kind': 'dropout',  'rate': 0.1},
}
p = run_factor_probe(
    config.replace(dataset='FD001', sensor_columns=None),
    factor='noise', levels=NOISE_LEVELS,
    models=[MODEL], baselines=BASELINES, device=device,
    out_csv=f'{RESULTS}/probe_noise_{TAG}.csv')
print('noise probe →', p)


## RQ-C — what to record & keep (channel subsets, FD001)

The C-MAPSS chapter of RQ-C (the fleet-scale version lands on N-CMAPSS/Backblaze in
Phase B). Levels — DECISION recorded here: `all21` = record everything **including the
FD001-constant channels** (junk-channel tolerance); `default14` = the recorded
non-constant default *(cache hit)*; `top8`/`min4` = progressively cheaper installs
keeping the high-signal sensors most cited by C-MAPSS feature-selection practice.


In [ ]:
from src.config import SENSOR_COLUMNS, FD001_NONCONSTANT_SENSORS

CHANNEL_LEVELS = {
    'all21':     list(SENSOR_COLUMNS),               # record everything (incl. constants)
    'default14': list(FD001_NONCONSTANT_SENSORS),    # cache hit — the recorded default
    'top8':      ['s_2', 's_3', 's_4', 's_7', 's_11', 's_12', 's_15', 's_21'],
    'min4':      ['s_4', 's_7', 's_11', 's_12'],
}
p = run_factor_probe(
    config.replace(dataset='FD001', sensor_columns=None),
    factor='channels', levels=CHANNEL_LEVELS,
    models=[MODEL], baselines=BASELINES, device=device,
    out_csv=f'{RESULTS}/probe_channels_{TAG}.csv')
print('channels probe →', p)


## RQ-E — how to label: the `max_rul` cap arm (FD001 + FD004), MSE + CORN

`cap125` is the recorded protocol *(cache hit)*; `cap200` raises the piecewise cap
(§18 machinery) — one new cache per dataset. Both losses run so the ordinal arm's
labeling interaction is captured. **Caveat:** clipped metrics are computed against each
level's OWN cap, so cross-cap comparison must use the `*_unclipped` columns; within-cap
cells score normally.


In [ ]:
CAP_LEVELS = {'cap125': {'max_rul': 125}, 'cap200': {'max_rul': 200}}
for ds in ['FD001', 'FD004']:
    p = run_factor_probe(
        config.replace(dataset=ds, sensor_columns=None),
        factor='label_cap', levels=CAP_LEVELS,
        models=[MODEL], baselines=BASELINES, device=device,
        losses=['mse', 'corn'],
        out_csv=f'{RESULTS}/probe_label_cap_{TAG}.csv')
    print('label-cap probe →', p)


## RQ-A — history / context length (FD004 anchor + the FD001 §12-caveat grid)

FD004 is the RQ-A anchor (RESEARCH_PLAN §5); FD001 re-runs the finer grid the §12
caveat asked for (median train history ≈ 199, so 256 ≈ "all available history").
`ctx256` is a *(cache hit)*; the other four levels each build one new cache per
dataset — **this is the long section** (≈ 8 embedding passes). Fully restartable.
Baselines don't read the TSFM context, so their rows repeat per level by design
(the win-rule bar is per-cell).


In [ ]:
CONTEXT_LEVELS = {f'ctx{n}': {'tsfm_context_length': n} for n in (32, 64, 128, 192, 256)}
for ds in ['FD001', 'FD004']:
    p = run_factor_probe(
        config.replace(dataset=ds, sensor_columns=None),
        factor='context', levels=CONTEXT_LEVELS,
        models=[MODEL], baselines=BASELINES, device=device,
        out_csv=f'{RESULTS}/probe_context_{TAG}.csv')
    print('context probe →', p)


In [ ]:
# Peek: seed-mean of every CSV this session wrote (scoring proper happens later on a
# core runtime — success_map over probe_*.csv with
# cell_fields=('dataset', 'n_units', 'factor', 'level'), globbing BOTH sessions' files).
import glob
import pandas as pd

files = sorted(glob.glob(f'{RESULTS}/probe_*_{TAG}.csv'))
files += sorted(glob.glob(f'{RESULTS}/representation_fairness_{TAG}.csv'))
for f in files:
    df = pd.read_csv(f)
    keys = [c for c in ('dataset', 'factor', 'level', 'mode') if c in df.columns]
    print('\n==', f.split('/')[-1], f'({len(df)} rows) ==')
    print(df.groupby(keys + ['model', 'loss'])[['rmse_clipped', 'nasa_clipped']]
            .mean().round(2).to_string())
